# SafeShift — openbmb/MiniCPM-V-4 — preflight không tiêu lượt

Sau review/merge, attach đúng bundle160 TRAIN và checkpoint Dataset hoặc Notebook
Output `notebooks/<owner>/<slug>/`. Chọn Internet ON và T4 ×2, Run All notebook
preflight này. Có thể chạy lại; không tạo execution/Owner slot, không tải/nạp
weights và không gọi mô hình. Hash index/shards đọc từ Input; CPU native empty
meta/processor dùng ảnh tổng hợp; child CUDA FP16 chỉ là phép toán tensor.

Python3.13 kernel chỉ điều phối. Worker CPython3.11.9 tự provision từ archive
python-build-standalone20240726 đã khóa URL/size/SHA256; không cần cài thủ công.
Software pins/cu124/precision/model revision/C1/parser/HNLO giữ nguyên.
Kiểm đồng thời cấp phát thật28 GiB scratch +4 GiB output; dung lượng free không
thay thế bằng chứng cấp phát. Không có CUDA/model-fit PASS từ CPU local.

**Cách chạy diagnostic trước thực nghiệm:** tạo notebook Kaggle riêng bằng file
này, attach cùng bundle/checkpoint Inputs của run160; T4 ×2, Internet ON.
Chạy Run All/Save & Run All của preflight, không mở notebook160 cùng lúc.
Preflight đọc/hash/decode Inputs, tải software pinned, cấp phát tạm28+4 GiB và
xóa scratch thuộc session; giữ evidence trong `data/processed/candidate160_preflight`.
Không cần grant/CONTINUE hoặc approval lặp lại trong phạm vi Owner đã cấp.

Download cả **ZIP**, **ZIP.sha256.json** và **.preflight-result.json** được in.
Receipt cuối `.preflight-result.json` phải có `evidence_export=VERIFIED`;
`preflight.json` trong ZIP còn ghi export PENDING vì được seal trước export.
Thiếu receipt cuối/checksum sai/export lỗi/partial → BLOCKED, giữ mọi file.
Receipt ghi từng check PASS/BLOCKED/NOT_RUN và tách ENVIRONMENT_OBSERVED,
CPU_SOFTWARE_PASS, LIVE_GPU_PREFLIGHT_PASS, WEIGHTS_LOADED, INFERENCE_VERIFIED,
SCIENTIFIC_QUALIFIED. Ba mức cuối vẫn NOT_RUN/UNVERIFIED trong preflight.
Chỉ PRELOAD_CHECKS_PASS_MODEL_UNVERIFIED khi mọi cổng được kiểm và export xác minh;
không READY hoặc bảo đảm peak memory/quality. Không tự chạy notebook160 sau đó.
Run160 vẫn kiểm lại mọi admission trước consumption vì tài nguyên có thể đổi.
Preflight không cấp quyền replay một lượt đã consumed/unknown.

Nếu cần thông tin Kaggle thực tế, gửi status/checks/levels và lỗi từ receipt
cuối hoặc ZIP private cho người review; không publish ảnh, weights hoặc secrets.
Code/local CPU PASS không phải GPU PASS; HEAD mới vẫn cần ChatGPT kiểm GitHub
và Antigravity audit độc lập trước merge. Quy tắc mới chưa COMPLETE trên main.

Runtime `b5448d031b287bc7adabd58ec36d98e671449b9d`, source SHA256 `09d94fd27a6a5260526c981a55854e8217536e0499824400ea9df5d213fb2d08`.


In [ ]:
import pathlib, json, hashlib, zipfile, uuid
try:
    # Generated by scripts/build_three_vlm_notebooks.py; do not edit.
    import base64, hashlib, json, pathlib, sys, zlib
    payload = json.loads(zlib.decompress(base64.b64decode('...')))
    expected_source_sha256 = '09d94fd27a6a5260526c981a55854e8217536e0499824400ea9df5d213fb2d08'
    inventory = {p: hashlib.sha256(t.encode()).hexdigest() for p,t in payload.items()}
    actual = hashlib.sha256((json.dumps(inventory,sort_keys=True,ensure_ascii=True,separators=(",",":"),allow_nan=False)+"\n").encode()).hexdigest()
    if actual != expected_source_sha256:
        raise RuntimeError("EMBEDDED_SOURCE_CORRUPT")
    root = pathlib.Path.cwd() / "data/processed/candidate160_preflight_code" / actual
    if any(p.is_symlink() for p in (root, *root.parents)):
        raise RuntimeError("EMBEDDED_ROOT_ALIAS")
    root.mkdir(parents=True, exist_ok=True)
    for name, text in payload.items():
        p = root / name
        if pathlib.PurePosixPath(name).is_absolute() or ".." in pathlib.PurePosixPath(name).parts:
            raise RuntimeError("UNSAFE_EMBEDDED_PATH")
        if any(parent.is_symlink() for parent in (p, *p.parents)):
            raise RuntimeError("EMBEDDED_PATH_ALIAS")
        p.parent.mkdir(parents=True,exist_ok=True)
        if p.exists():
            if p.read_bytes() != text.encode():
                raise RuntimeError("EXISTING_SOURCE_CHANGED")
        else:
            with p.open("xb") as stream:
                stream.write(text.encode())
    sys.path.insert(0,str(root))
    if "safeshift" in sys.modules and not pathlib.Path(sys.modules["safeshift"].__file__).resolve().is_relative_to(root.resolve()):
        raise RuntimeError("STALE_IMPORTED_SAFESHIFT_REQUIRES_CLEAN_SESSION")
    from safeshift.candidates.train_preflight import launch_preflight as launch
except BaseException as error:
    directory = (pathlib.Path.cwd() / "data/processed/candidate160_bootstrap") / uuid.uuid4().hex
    directory.mkdir(parents=True, exist_ok=False)
    record = {"status":"BLOCKED", "stage":"notebook_bootstrap", "attempt_consumed":False,
              "type":type(error).__name__, "constraint":str(error)[:300],
              "source_sha256":globals().get("expected_source_sha256"),
              "cuda":"NOT_CALLED", "weights":"NOT_REQUESTED"}
    payload_bytes = (json.dumps(record, sort_keys=True)+"\n").encode()
    (directory / "failure.json").write_bytes(payload_bytes)
    hashes = {"failure.json":hashlib.sha256(payload_bytes).hexdigest()}
    (directory / "checksums.json").write_text(json.dumps(hashes), encoding="utf-8")
    archive = directory.with_suffix(".zip")
    with zipfile.ZipFile(archive,"x",compression=zipfile.ZIP_DEFLATED) as packed:
        for name in ("failure.json","checksums.json"):
            packed.write(directory / name,name)
    (pathlib.Path(str(archive)+".sha256.json")).write_text(json.dumps({"sha256":hashlib.sha256(archive.read_bytes()).hexdigest()}),encoding="utf-8")
    print("BLOCKED before attempt: download", archive, "and its .sha256.json")
    raise
evidence = launch('minicpm4', expected_source_sha256, root, 'b5448d031b287bc7adabd58ec36d98e671449b9d')
print("Evidence directory:", evidence)
